In [1]:
ACTIONS = {
    "noop": 0,

    "button_up": 1,
    "button_down": 2,

    "text_up": 3,
    "text_down": 4,

    "font_up": 5,
    "font_down": 6,

    "spacing_up": 7,
    "spacing_down": 8,

    "enable_tooltips": 9,
}


In [2]:
import random
import csv


In [3]:
def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))

def rand(a, b):
    return a + random.random() * (b - a)


In [4]:
def sample_persona():
    r = random.random()
    if r < 0.5:
        return "novice_old"
    elif r < 0.8:
        return "intermediate"
    return "expert"


In [5]:
def generate_state(persona):
    if persona == "novice_old":
        s = [
            rand(0.5, 1.0), rand(0.6, 1.0), rand(0.5, 0.9), rand(0.4, 0.8),
            rand(0.6, 1.0), rand(0.2, 0.5), rand(0.6, 1.0), rand(0.3, 0.7),
            rand(0.6, 1.0), rand(0.3, 0.7), rand(0.6, 1.0), rand(0.6, 1.0)
        ]
    elif persona == "intermediate":
        s = [rand(0.3, 0.7) for _ in range(12)]
    else:
        s = [
            rand(0.1, 0.4), rand(0.1, 0.4), rand(0.1, 0.4), rand(0.1, 0.4),
            rand(0.1, 0.4), rand(0.6, 1.0), rand(0.1, 0.3), rand(0.1, 0.3),
            rand(0.1, 0.3), rand(0.1, 0.3), rand(0.1, 0.3), rand(0.1, 0.3)
        ]

    persona_onehot = [
        1.0 if persona == "novice_old" else 0.0,
        1.0 if persona == "intermediate" else 0.0,
        1.0 if persona == "expert" else 0.0,
    ]

    return s + persona_onehot


In [6]:
def generate_ui_levels():
    levels = [random.randint(0, 6) for _ in range(4)]
    return levels, [lvl / 6.0 for lvl in levels]


In [7]:
def compute_scores(s):
    motor = s[6] + s[8] + s[11]
    cognitive = s[4] + s[0]
    nav = s[1] + s[2] + s[10]
    efficiency = s[5] - s[6] - s[4]
    return motor, cognitive, nav, efficiency


In [8]:
def rank_actions(motor, cognitive, nav, efficiency, persona):
    if cognitive > 1.4 and persona != "expert":
        return ["enable_tooltips", "font_up", "text_up"]

    if motor > 1.6:
        return ["button_up", "spacing_up", "font_up"]

    if nav > 1.8:
        return ["spacing_up", "text_up", "enable_tooltips"]

    if efficiency > 0.3 and persona == "expert":
        return ["noop", "spacing_down", "button_down"]

    return ["noop", "spacing_up", "font_up"]


In [9]:
def filter_actions(actions, ui):
    btn, txt, spacing, font = ui
    valid = []

    for a in actions:
        if a == "button_up" and btn == 6: continue
        if a == "button_down" and btn == 0: continue
        if a == "text_up" and txt == 6: continue
        if a == "text_down" and txt == 0: continue
        if a == "spacing_up" and spacing == 6: continue
        if a == "spacing_down" and spacing == 0: continue
        if a == "font_up" and font == 6: continue
        if a == "font_down" and font == 0: continue
        valid.append(a)

    return valid if valid else ["noop"]


In [ ]:
def choose_action(actions):
    """
    Select action from ranked list with reward based on quality.
    ALIGNED WITH FRONTEND: taskReward.jsx uses +0.6 completion, -0.4 timeout, -0.02 × pathLength
    This function bridges synthetic data generation with frontend reward weights.
    """
    r = random.random()
    if len(actions) == 1:
        return actions[0], 1.0  # Only option gets best reward

    # Action ranking probabilities (unchanged - matches frontend exploration)
    if r < 0.55:
        return actions[0], 1.0        # Best ranked action (55%)
    elif r < 0.85:
        return actions[min(1, len(actions)-1)], 0.85  # Second ranked (30%)
    else:
        return actions[min(2, len(actions)-1)], 0.65  # Third ranked (15%) - ALIGNED WITH FRONTEND


In [ ]:
def next_state(s, ui, action):
    """
    Apply state transition with improvements/degradations based on action.
    ALIGNED WITH FRONTEND: taskReward.jsx path penalty propagates through metric degradation
    Inefficient actions (down actions) degrade metrics as if adding path complexity.
    """
    ns = s[:]

    # small drift
    ns[4] = clamp(ns[4] + rand(-0.05, 0.05))
    ns[6] = clamp(ns[6] + rand(-0.05, 0.05))
    ns[0] = clamp(ns[0] + rand(0.01, 0.03))

    # improvement simulation
    if action in ["button_up", "spacing_up"]:
        ns[6] = clamp(ns[6] - 0.05)
        ns[11] = clamp(ns[11] - 0.05)

    if action == "enable_tooltips":
        ns[4] = clamp(ns[4] - 0.08)

    # ALIGNED WITH FRONTEND: Penalty for inefficient "down" actions
    # Frontend applies -0.02 × pathLength; we simulate this via metric degradation
    # "Down" actions (reductions) are less efficient, adding to interaction cost
    if action in ["button_down", "text_down", "font_down", "spacing_down"]:
        # Degrade efficiency metric to reflect path complexity
        ns[0] = clamp(ns[0] - 0.02)

    btn, txt, spacing, font = ui

    if action == "button_up": btn += 1
    if action == "button_down": btn -= 1
    if action == "text_up": txt += 1
    if action == "text_down": txt -= 1
    if action == "spacing_up": spacing += 1
    if action == "spacing_down": spacing -= 1
    if action == "font_up": font += 1
    if action == "font_down": font -= 1

    btn = max(0, min(6, btn))
    txt = max(0, min(6, txt))
    spacing = max(0, min(6, spacing))
    font = max(0, min(6, font))

    return ns, [btn, txt, spacing, font]


In [12]:
def is_done(s):
    return 1 if s[0] > 0.95 else 0


In [13]:
def generate_dataset(N=100000, filename="synthetic_rl_dataset.csv"):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)

        for _ in range(N):
            persona = sample_persona()
            s = generate_state(persona)

            ui_levels, ui_norm = generate_ui_levels()

            motor, cog, nav, eff = compute_scores(s)
            ranked = rank_actions(motor, cog, nav, eff, persona)
            ranked = filter_actions(ranked, ui_levels)

            action_name, reward = choose_action(ranked)

            ns, new_ui = next_state(s, ui_levels, action_name)

            row = (
                s + ui_norm +
                [ACTIONS[action_name], reward] +
                ns + [lvl / 6.0 for lvl in new_ui] +
                [is_done(ns)]
            )

            writer.writerow(row)

    print("Dataset generated:", filename)


In [14]:
generate_dataset()


Dataset generated: synthetic_rl_dataset.csv


In [33]:
import pandas as pd
import numpy as np

real_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Feb/dqn_ui_dataset_1769879573157.csv")

print("Real shape:", real_df.shape)
print(real_df.head())


Real shape: (608, 33)
   s_session_duration  s_total_distance  s_num_actions  s_num_clicks  \
0              6.6278          0.000000            0.0           1.0   
1             18.8523        326.323421            3.0           4.0   
2             27.2920       1802.283566            6.0           7.0   
3             42.3241       3128.793791           10.0          11.0   
4             53.2761       3434.016722           14.0          15.0   

   s_mean_time_per_action  s_vel_mean   s_vel_std  s_accel_mean  s_accel_std  \
0                0.000000    0.000000    0.000000      0.000000     0.000000   
1                4.074833   53.784393   46.062314     45.882618    83.236540   
2                3.444033  174.382874  197.934140    107.359175   144.873846   
3                3.569630  160.743694  179.290144     73.963053   132.936259   
4                3.332021  127.543677  161.413248     52.149040   122.206173   

   s_curve_mean  ...  next_s_vel_std  next_s_accel_mean  next_s_

In [34]:
real_oversampled = pd.concat([real_df]*10, ignore_index=True)
print(len(real_oversampled))  # ~6000


6080


In [35]:
def add_small_noise(df, feature_cols):
    noisy = df.copy()
    for col in feature_cols:
        noisy[col] += np.random.normal(0, 0.01, size=len(df))
        noisy[col] = noisy[col].clip(0, 1)
    return noisy

state_feature_cols = real_oversampled.columns[:19]

real_augmented = add_small_noise(real_oversampled, state_feature_cols)

print("After noise:", real_augmented.shape)


After noise: (6080, 33)


In [36]:
synthetic_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Feb/synthetic_rl_dataset.csv",
    header=None
)

print("Synthetic raw shape:", synthetic_df.shape)


Synthetic raw shape: (100000, 41)


In [37]:
state_cols = [
    "s_session_duration", "s_total_distance", "s_num_actions",
    "s_num_clicks", "s_mean_time_per_action",
    "s_vel_mean", "s_vel_std",
    "s_accel_mean", "s_accel_std",
    "s_curve_mean", "s_curve_std",
    "s_jerk_mean",
    "s_persona_novice_old",
    "s_persona_intermediate",
    "s_persona_expert",
    "s_btn_level", "s_text_level",
    "s_spacing_level", "s_font_level"
]

next_state_cols = [
    "next_s_session_duration", "next_s_total_distance", "next_s_num_actions",
    "next_s_num_clicks", "next_s_mean_time_per_action",
    "next_s_vel_mean", "next_s_vel_std",
    "next_s_accel_mean", "next_s_accel_std",
    "next_s_curve_mean", "next_s_curve_std",
    "next_s_jerk_mean",
    "next_s_persona_novice_old",
    "next_s_persona_intermediate",
    "next_s_persona_expert",
    "next_s_btn_level", "next_s_text_level",
    "next_s_spacing_level", "next_s_font_level"
]

all_columns = state_cols + ["action", "reward"] + next_state_cols + ["done"]

synthetic_df.columns = all_columns

print("Synthetic renamed shape:", synthetic_df.shape)


Synthetic renamed shape: (100000, 41)


In [38]:
missing_ui_cols = [
    "s_btn_level","s_text_level","s_spacing_level","s_font_level",
    "next_s_btn_level","next_s_text_level","next_s_spacing_level","next_s_font_level"
]

for col in missing_ui_cols:
    real_augmented[col] = 0.5


In [39]:
real_augmented = real_augmented[all_columns]

print("Real aligned shape:", real_augmented.shape)


Real aligned shape: (6080, 41)


In [40]:
final_df = pd.concat([synthetic_df, real_augmented], ignore_index=True)

final_df = final_df.sample(frac=1).reset_index(drop=True)

print("Final shape:", final_df.shape)


Final shape: (106080, 41)


In [41]:
print("Total NaNs:", final_df.isna().sum().sum())
print("Action unique values:", sorted(final_df["action"].unique()))
print("Action min:", final_df["action"].min())
print("Action max:", final_df["action"].max())


Total NaNs: 0
Action unique values: [np.float64(0.0), np.float64(2.458481378843951e-05), np.float64(3.310692394592824e-05), np.float64(3.7434780511594654e-05), np.float64(4.634601380936619e-05), np.float64(7.658169386680327e-05), np.float64(9.835525086344699e-05), np.float64(0.00010719995589593462), np.float64(0.00011716238323509287), np.float64(0.00011982502343009281), np.float64(0.00012396693443038947), np.float64(0.0001266065892244385), np.float64(0.0001710607971639536), np.float64(0.00018261572894518786), np.float64(0.0001925869734447274), np.float64(0.0002474781253116172), np.float64(0.00025760624330312804), np.float64(0.00026864288980827495), np.float64(0.00029834608406408795), np.float64(0.00031118081204444014), np.float64(0.0003191162594293566), np.float64(0.0003240305474328318), np.float64(0.0003283747792016029), np.float64(0.0003336359106737024), np.float64(0.00034299619460166647), np.float64(0.0003558907558388804), np.float64(0.00043085644248307955), np.float64(0.00043483046

In [42]:
final_df["action"] = final_df["action"].round().astype(int)


In [43]:
final_df = final_df[
    (final_df["action"] >= 0) &
    (final_df["action"] <= 9)
]


In [44]:
print("Action unique:", sorted(final_df["action"].unique()))
print("Action min:", final_df["action"].min())
print("Action max:", final_df["action"].max())


Action unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(8), np.int64(9)]
Action min: 0
Action max: 9


In [45]:
state_feature_cols = real_oversampled.columns[:19]
